# 07D — Promote the reviewed end-to-end thresholds

## Objective

Notebook 07C produced a candidate threshold set calibrated with the **actual deployed detector rule**:

`hypothesis == class AND LLR_class >= threshold_class`.

This notebook performs the small, explicit promotion step:

1. verify the 07C candidate files exist;
2. verify all detector classes are present;
3. verify the candidate metadata says no GMMs were retrained;
4. archive the current threshold file;
5. replace only `thresholds.json`;
6. update `detector_metadata.json` with the final calibration rule and provenance;
7. verify that no GMM file changed.

No model fitting, feature extraction, calibration search, or field-audio processing happens here.

In [1]:
from pathlib import Path
import hashlib
import json
import shutil
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd

pd.set_option("display.width", 190)
pd.set_option("display.max_columns", 100)

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

MODEL_DIR = (
    REPO_ROOT
    / "models"
    / "gmm_robust_final"
)

CALIBRATION_DIR = (
    REPO_ROOT
    / "data"
    / "processed"
    / "final_detector_calibration"
)

CANDIDATE_DIR = (
    CALIBRATION_DIR
    / "end_to_end_threshold_calibration"
)

THRESHOLDS_PATH = (
    MODEL_DIR
    / "thresholds.json"
)

METADATA_PATH = (
    MODEL_DIR
    / "detector_metadata.json"
)

CANDIDATE_THRESHOLDS_PATH = (
    CANDIDATE_DIR
    / "candidate_thresholds.json"
)

CANDIDATE_METADATA_PATH = (
    CANDIDATE_DIR
    / "candidate_metadata.json"
)

CANDIDATE_METRICS_PATH = (
    CANDIDATE_DIR
    / "candidate_metrics_by_species.csv"
)

MACRO_COMPARISON_PATH = (
    CANDIDATE_DIR
    / "macro_before_after.csv"
)

for path in (
    MODEL_DIR,
    THRESHOLDS_PATH,
    METADATA_PATH,
    CANDIDATE_THRESHOLDS_PATH,
    CANDIDATE_METADATA_PATH,
    CANDIDATE_METRICS_PATH,
    MACRO_COMPARISON_PATH,
):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required input: {path}"
        )

print("Model directory   :", MODEL_DIR)
print("Candidate outputs :", CANDIDATE_DIR)

Model directory   : C:\Users\sofia\Projects\PUCE\frog-presence-replication\models\gmm_robust_final
Candidate outputs : C:\Users\sofia\Projects\PUCE\frog-presence-replication\data\processed\final_detector_calibration\end_to_end_threshold_calibration


## 2. Load current and candidate detector state

In [2]:
with open(
    THRESHOLDS_PATH,
    "r",
    encoding="utf-8",
) as f:
    old_thresholds = json.load(f)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8",
) as f:
    detector_metadata = json.load(f)

with open(
    CANDIDATE_THRESHOLDS_PATH,
    "r",
    encoding="utf-8",
) as f:
    candidate_thresholds = json.load(f)

with open(
    CANDIDATE_METADATA_PATH,
    "r",
    encoding="utf-8",
) as f:
    candidate_metadata = json.load(f)

candidate_metrics = pd.read_csv(
    CANDIDATE_METRICS_PATH
)

macro_comparison = pd.read_csv(
    MACRO_COMPARISON_PATH
)

classes = tuple(
    detector_metadata[
        "classes"
    ]
)

print("Feature        :", detector_metadata["feature"])
print("GMM components :", detector_metadata["n_components"])
print("Classes        :", classes)

threshold_comparison = pd.DataFrame([
    {
        "code": code,
        "old_threshold": float(
            old_thresholds[
                code
            ]
        ),
        "candidate_threshold": float(
            candidate_thresholds[
                code
            ]
        ),
        "change": (
            float(
                candidate_thresholds[
                    code
                ]
            )
            - float(
                old_thresholds[
                    code
                ]
            )
        ),
    }
    for code in classes
])

display(
    threshold_comparison.round(4)
)

display(
    macro_comparison.round(4)
)

Feature        : 20_PLP_RASTA
GMM components : 4
Classes        : ('f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f10')


,code,old_threshold,candidate_threshold,change
0,f01,8.0751,8.0751,0.0000
1,f02,8.5813,3.7012,-4.8801
2,f03,6.9382,6.9382,0.0000
3,f04,1.9181,5.0647,3.1467
4,f05,5.5535,5.5535,0.0000
5,f06,5.2310,5.2310,0.0000
6,f07,5.7501,5.7501,0.0000
7,f08,7.1894,7.1894,0.0000
8,f10,9.0146,9.0146,0.0000


,threshold_set,accepted_fraction,correct_and_accepted_fraction,macro_recall,macro_precision,macro_F1,macro_specificity,macro_FPR
0,old_07_one_vs_all,0.9372,0.9267,0.8955,0.9533,0.9070,0.9986,0.0014
1,new_07C_end_to_end,0.9895,0.9764,0.9834,0.9844,0.9838,0.9983,0.0017


## 3. Promotion safety checks

Promotion is allowed only if:

- candidate thresholds contain exactly the detector classes;
- every value is finite;
- 07C states that GMMs were not retrained;
- 07C states that scores were not recomputed;
- 07C states that the TRAIN/CALIBRATION split was unchanged;
- every reported candidate FPR is at most 5%.

In [3]:
if set(candidate_thresholds) != set(classes):
    raise RuntimeError(
        "Candidate threshold classes do not exactly match detector classes."
    )

if not all(
    np.isfinite(
        float(value)
    )
    for value in candidate_thresholds.values()
):
    raise RuntimeError(
        "At least one candidate threshold is non-finite."
    )

required_candidate_flags = {
    "gmm_retrained": False,
    "scores_recomputed": False,
    "train_calibration_split_changed": False,
    "old_threshold_file_overwritten": False,
    "review_required_before_promotion": True,
}

for key, expected in required_candidate_flags.items():
    actual = candidate_metadata.get(
        key,
        None,
    )

    if actual != expected:
        raise RuntimeError(
            f"Candidate metadata check failed for {key}: "
            f"expected {expected!r}, got {actual!r}"
        )

if (
    candidate_metrics[
        "FPR"
    ] > 0.05 + 1e-12
).any():
    raise RuntimeError(
        "At least one candidate per-species FPR exceeds 5%."
    )

print("Class set                 : OK")
print("Finite thresholds         : OK")
print("No GMM retraining         : OK")
print("No score recomputation    : OK")
print("Split unchanged           : OK")
print("Candidate FPR <= 5%       : OK")

Class set                 : OK
Finite thresholds         : OK
No GMM retraining         : OK
No score recomputation    : OK
Split unchanged           : OK
Candidate FPR <= 5%       : OK


## 4. Hash all frozen GMM files before promotion

Only JSON metadata/threshold files are allowed to change.

In [4]:
def sha256_file(path):
    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:
        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


gmm_paths = sorted(
    MODEL_DIR.glob(
        "f*.joblib"
    )
)

if not gmm_paths:
    raise RuntimeError(
        "No frozen GMM .joblib files found."
    )

gmm_hashes_before = {
    path.name: sha256_file(
        path
    )
    for path in gmm_paths
}

display(
    pd.DataFrame([
        {
            "model_file": name,
            "sha256": digest,
        }
        for name, digest
        in gmm_hashes_before.items()
    ])
)

,model_file,sha256
0,f01.joblib,51b19a63c6efeefffbd4b393f8350b793104cf5007050d...
1,f02.joblib,90eeac565349453e7fcb4ee57bbde193dc2e912e73c863...
2,f03.joblib,3435a335c52fa349560aafb76f312bd4ba700b5bcf9896...
3,f04.joblib,865becdf6ab92840cd67d718a4c7baf0acbd9ac77454d6...
4,f05.joblib,778d90cb7ce6ae36cd48912eef77ad633827c1473f35fb...
5,f06.joblib,4664aa33acbdc0f66303104ae83a59c4495ce3fb411315...
6,f07.joblib,5f687027e950761556625abe49c57f9dc524492d04ff4b...
7,f08.joblib,989ea158ca6e1234a660b5ad67caae8b649289cdea020e...
8,f10.joblib,f626452a7c58c3809afb5624abbe3c78d9b6331f9caa01...


## 5. Archive current thresholds and metadata

The previous 07 threshold set is retained for provenance.

In [5]:
ARCHIVE_DIR = (
    MODEL_DIR
    / "archive"
)

ARCHIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

archive_tag = (
    "pre_07C_end_to_end"
)

threshold_archive_path = (
    ARCHIVE_DIR
    / f"thresholds_{archive_tag}.json"
)

metadata_archive_path = (
    ARCHIVE_DIR
    / f"detector_metadata_{archive_tag}.json"
)

if not threshold_archive_path.exists():
    shutil.copy2(
        THRESHOLDS_PATH,
        threshold_archive_path,
    )

if not metadata_archive_path.exists():
    shutil.copy2(
        METADATA_PATH,
        metadata_archive_path,
    )

print("Archived thresholds:", threshold_archive_path)
print("Archived metadata  :", metadata_archive_path)

Archived thresholds: C:\Users\sofia\Projects\PUCE\frog-presence-replication\models\gmm_robust_final\archive\thresholds_pre_07C_end_to_end.json
Archived metadata  : C:\Users\sofia\Projects\PUCE\frog-presence-replication\models\gmm_robust_final\archive\detector_metadata_pre_07C_end_to_end.json


## 6. Promote candidate thresholds

In [6]:
with open(
    THRESHOLDS_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            code: float(
                candidate_thresholds[
                    code
                ]
            )
            for code in classes
        },
        f,
        indent=2,
    )

print(
    "Promoted candidate thresholds to:",
    THRESHOLDS_PATH,
)

Promoted candidate thresholds to: C:\Users\sofia\Projects\PUCE\frog-presence-replication\models\gmm_robust_final\thresholds.json


## 7. Update detector metadata

The metadata records that threshold calibration is now **end-to-end** rather than the earlier raw
one-vs-all ROC criterion.

In [7]:
updated_metadata = dict(
    detector_metadata
)

updated_metadata[
    "threshold_calibration"
] = {
    "status":
        "final_promoted",
    "method":
        (
            "end-to-end per class: "
            "hypothesis==class AND LLR_class>=threshold"
        ),
    "selection_rule":
        (
            "maximize calibration recall subject to "
            "per-species FPR<=0.05; larger threshold breaks recall ties"
        ),
    "source_notebook":
        "07C_final_threshold_calibration",
    "candidate_threshold_source":
        str(
            CANDIDATE_THRESHOLDS_PATH
        ),
    "gmm_retrained_after_calibration":
        False,
    "scores_recomputed_after_calibration":
        False,
    "train_calibration_split_changed":
        False,
    "previous_thresholds_archived_at":
        str(
            threshold_archive_path
        ),
}

updated_metadata[
    "thresholds"
] = {
    code: float(
        candidate_thresholds[
            code
        ]
    )
    for code in classes
}

updated_metadata[
    "calibration_interpretation"
] = (
    "Calibration metrics are not independent test metrics. "
    "Calibration negatives are other modelled frog calls, not a complete "
    "sample of environmental noise or unmodelled species."
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        updated_metadata,
        f,
        indent=2,
    )

print(
    "Updated detector metadata:",
    METADATA_PATH,
)

Updated detector metadata: C:\Users\sofia\Projects\PUCE\frog-presence-replication\models\gmm_robust_final\detector_metadata.json


## 8. Verify promoted detector state

In [8]:
with open(
    THRESHOLDS_PATH,
    "r",
    encoding="utf-8",
) as f:
    promoted_thresholds = json.load(f)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8",
) as f:
    promoted_metadata = json.load(f)

assert set(
    promoted_thresholds
) == set(
    classes
)

for code in classes:
    assert np.isclose(
        float(
            promoted_thresholds[
                code
            ]
        ),
        float(
            candidate_thresholds[
                code
            ]
        ),
    )

assert (
    promoted_metadata[
        "threshold_calibration"
    ][
        "status"
    ]
    == "final_promoted"
)

print("Promoted thresholds:")
display(
    pd.DataFrame([
        {
            "code": code,
            "threshold": float(
                promoted_thresholds[
                    code
                ]
            ),
        }
        for code in classes
    ]).round(4)
)

Promoted thresholds:


,code,threshold
0,f01,8.0751
1,f02,3.7012
2,f03,6.9382
3,f04,5.0647
4,f05,5.5535
5,f06,5.2310
6,f07,5.7501
7,f08,7.1894
8,f10,9.0146


## 9. Verify that frozen GMMs did not change

In [9]:
gmm_hashes_after = {
    path.name: sha256_file(
        path
    )
    for path in gmm_paths
}

if (
    gmm_hashes_before
    != gmm_hashes_after
):
    raise RuntimeError(
        "At least one frozen GMM changed during threshold promotion."
    )

print(
    "Frozen GMM hashes unchanged: OK"
)

Frozen GMM hashes unchanged: OK


## 10. Final detector manifest

In [10]:
final_manifest = {
    "detector_status":
        "frozen_after_07C_threshold_promotion",
    "feature":
        promoted_metadata[
            "feature"
        ],
    "feature_dimensions":
        promoted_metadata[
            "feature_dimensions"
        ],
    "n_components":
        promoted_metadata[
            "n_components"
        ],
    "classes":
        list(
            classes
        ),
    "thresholds":
        {
            code: float(
                promoted_thresholds[
                    code
                ]
            )
            for code in classes
        },
    "threshold_rule":
        (
            "accept winning hypothesis k iff "
            "LLR_k >= threshold_k"
        ),
    "llr_definition":
        promoted_metadata[
            "llr_definition"
        ],
    "validation_unit":
        "source_wav",
    "gmm_files":
        [
            path.name
            for path in gmm_paths
        ],
    "gmm_sha256":
        gmm_hashes_after,
    "previous_threshold_archive":
        str(
            threshold_archive_path
        ),
    "limitations":
        promoted_metadata.get(
            "evaluation_limitations",
            [],
        ),
}

MANIFEST_PATH = (
    MODEL_DIR
    / "final_detector_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_manifest,
        f,
        indent=2,
    )

print(
    json.dumps(
        final_manifest,
        indent=2,
    )
)

{
  "detector_status": "frozen_after_07C_threshold_promotion",
  "feature": "20_PLP_RASTA",
  "feature_dimensions": 20,
  "n_components": 4,
  "classes": [
    "f01",
    "f02",
    "f03",
    "f04",
    "f05",
    "f06",
    "f07",
    "f08",
    "f10"
  ],
  "thresholds": {
    "f01": 8.075132278238918,
    "f02": 3.701206002175951,
    "f03": 6.938241728577303,
    "f04": 5.064719360802286,
    "f05": 5.553542470065025,
    "f06": 5.231028875936364,
    "f07": 5.750085001119672,
    "f08": 7.189361271867611,
    "f10": 9.01463416154914
  },
  "threshold_rule": "accept winning hypothesis k iff LLR_k >= threshold_k",
  "llr_definition": "class mean log-likelihood minus median mean log-likelihood of all alternative models",
  "validation_unit": "source_wav",
  "gmm_files": [
    "f01.joblib",
    "f02.joblib",
    "f03.joblib",
    "f04.joblib",
    "f05.joblib",
    "f06.joblib",
    "f07.joblib",
    "f08.joblib",
    "f10.joblib"
  ],
  "gmm_sha256": {
    "f01.joblib": "51b19a63c6e

## 11. Final status

After this notebook completes successfully:

### Frozen model

- feature: **20 PLP-RASTA**
- diagonal GMM order: **M = 4**
- one frozen GMM per usable species
- `f09` excluded because the recovered labelled material is insufficient

### Frozen verification

- one LLR threshold per species
- thresholds calibrated with the actual end-to-end detector rule
- per-species calibration FPR constrained to at most 5%
- previous thresholds archived
- GMM binary hashes verified unchanged

### Next stage

The statistical classifier is now frozen.

The next separate problem is **candidate-call segmentation in long recordings**.

That frontend must produce candidate acoustic events, after which the frozen detector can classify and
verify them. Segmentation should be evaluated independently before we start reporting field
presence/absence results, because otherwise one badly chosen energy threshold can make several weeks of
careful model validation look decorative.